# Data Preparation for Military Aircraft Detection
This notebook processes the `military-aircraft-recognition-dataset` to create a dataset suitable for training an Oriented Bounding Box (OBB) model (e.g., YOLOv8-OBB).
It will:
1. Parse the XML annotations (Oriented Bounding Boxes).
2. Create a train / validation / test split.
3. Convert the annotations to YOLO OBB format (`class_index x1 y1 x2 y2 x3 y3 x4 y4` normalized).
4. Save the images and labels into the `train`, `validation`, and `test` directories.

In [7]:
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split

In [8]:
# Define paths
data_dir = Path("data")
dataset_dir = data_dir / "military-aircraft-recognition-dataset"
images_dir = dataset_dir / "JPEGImages"
annotations_dir = dataset_dir / "Annotations" / "Oriented Bounding Boxes"

# Output directories
splits = ["train", "validation", "test"]
for split in splits:
    (data_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (data_dir / split / "labels").mkdir(parents=True, exist_ok=True)

In [ ]:
# Check if dataset exists, if not download it
import os

if not dataset_dir.exists():
    print("Dataset not found. Downloading from Kaggle...")
    # Use curl to download the zip file
    download_cmd = "curl -L -o data/military-aircraft-recognition-dataset.zip https://www.kaggle.com/api/v1/datasets/download/khlaifiabilel/military-aircraft-recognition-dataset"
    exit_code = os.system(download_cmd)

    if exit_code == 0:
        print("Download successful. Extracting...")
        os.system(
            "unzip -q data/military-aircraft-recognition-dataset.zip -d data/military-aircraft-recognition-dataset"
        )
        print("Extraction complete!")
    else:
        print(
            "Error: Failed to download the dataset. You might need to authenticate with Kaggle API."
        )
else:
    print("Dataset already exists. Ready to process!")


Dataset already exists. Ready to process!


In [ ]:
# Define classes and class to index mapping
classes = [
    "A1",
    "A2",
    "A3",
    "A4",
    "A5",
    "A6",
    "A7",
    "A8",
    "A9",
    "A10",
    "A11",
    "A12",
    "A13",
    "A14",
    "A15",
    "A16",
    "A17",
    "A18",
    "A19",
    "A20",
]
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

# Create data.yaml for YOLO OBB
dataset_path = data_dir.absolute().as_posix()
yaml_content = """path: data
train: train/images
val: validation/images
test: test/images

names:
"""
for idx, cls in enumerate(classes):
    yaml_content += f"  {idx}: {cls}\n"

with open(data_dir / "data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
# Get all valid files and create splits
all_files = [f.stem for f in annotations_dir.glob("*.xml")]

# Split data into train (80%), validation (10%), and test (10%) using sklearn
train_files, temp_files = train_test_split(all_files, test_size=0.2, random_state=6)
val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=6
)  # 1/2 of 20% is 10%

split_dict = {"train": train_files, "validation": val_files, "test": test_files}

print(f"Total files: {len(all_files)}")
print(f"Train: {len(split_dict['train'])}")
print(f"Validation: {len(split_dict['validation'])}")
print(f"Test: {len(split_dict['test'])}")

Total files: 3842
Train: 3073
Validation: 384
Test: 385


In [13]:
# Function to convert XML to YOLO OBB format
def convert_xml_to_yolo_obb(xml_path, img_path, output_txt_path, class_to_idx):
    def clamp01(v):
        return max(0.0, min(1.0, float(v)))

    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    width = float(size.find("width").text)
    height = float(size.find("height").text)

    # Fallback to reading actual image dimensions if XML is broken
    if width <= 0 or height <= 0:
        with Image.open(img_path) as img:
            width, height = img.size

    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in class_to_idx:
            continue

        class_idx = class_to_idx[name]
        robndbox = obj.find("robndbox")

        # Get coordinates (pixel values from XML)
        x_lt = float(robndbox.find("x_left_top").text)
        y_lt = float(robndbox.find("y_left_top").text)
        x_rt = float(robndbox.find("x_right_top").text)
        y_rt = float(robndbox.find("y_right_top").text)
        x_rb = float(robndbox.find("x_right_bottom").text)
        y_rb = float(robndbox.find("y_right_bottom").text)
        x_lb = float(robndbox.find("x_left_bottom").text)
        y_lb = float(robndbox.find("y_left_bottom").text)

        # Heuristic: if coordinates look already normalized (<=1), keep; else normalize by width/height
        coords = [x_lt, y_lt, x_rt, y_rt, x_rb, y_rb, x_lb, y_lb]
        if any(c > 1.0 for c in coords):
            x_lt /= width
            y_lt /= height
            x_rt /= width
            y_rt /= height
            x_rb /= width
            y_rb /= height
            x_lb /= width
            y_lb /= height

        # Clamp to [0,1] to avoid tiny negatives or >1 rounding issues
        x_lt = clamp01(x_lt)
        y_lt = clamp01(y_lt)
        x_rt = clamp01(x_rt)
        y_rt = clamp01(y_rt)
        x_rb = clamp01(x_rb)
        y_rb = clamp01(y_rb)
        x_lb = clamp01(x_lb)
        y_lb = clamp01(y_lb)

        lines.append(
            f"{class_idx} {x_lt:.6f} {y_lt:.6f} {x_rt:.6f} {y_rt:.6f} {x_rb:.6f} {y_rb:.6f} {x_lb:.6f} {y_lb:.6f}"
        )

    with open(output_txt_path, "w") as f:
        f.write("\n".join(lines))


# Process all files
for split, file_list in split_dict.items():
    print(f"Processing {split} split...")
    for file_stem in tqdm(file_list):
        xml_path = annotations_dir / f"{file_stem}.xml"
        img_path = images_dir / f"{file_stem}.jpg"

        if not img_path.exists():
            print(f"Warning: Image not found {img_path}")
            continue

        # Copy image
        dst_img_path = data_dir / split / "images" / f"{file_stem}.jpg"
        shutil.copy(img_path, dst_img_path)

        # Convert and write labels
        dst_label_path = data_dir / split / "labels" / f"{file_stem}.txt"
        convert_xml_to_yolo_obb(xml_path, img_path, dst_label_path, class_to_idx)

print("Dataset preparation completed successfully!")

Processing train split...


100%|██████████| 3073/3073 [00:02<00:00, 1063.13it/s]


Processing validation split...


100%|██████████| 384/384 [00:00<00:00, 987.70it/s] 


Processing test split...


100%|██████████| 385/385 [00:00<00:00, 1026.99it/s]

Dataset preparation completed successfully!
